# N1 — Width × Noise Phase Diagram

**Core contribution of the paper.**

| | |
|---|---|
| Model | CNN5 (5-layer CNN, width multiplier k) |
| Dataset | CIFAR-10, n=5000, varying η |
| Sweep | k ∈ {2,4,8,16,32,64} × η ∈ {0%,5%,10%,20%,30%,40%} × 2 seeds = **72 runs** |
| Output | Fig 3 heatmap, Fig 4 DD overlay, Fig 5 phase diagram, benign boundary fit |

## Parallelisation (3 accounts)

| Account | Member | Set `MY_NOISE_RATES` to | Runs | Est. time |
|---------|--------|------------------------|------|-----------|
| **A** | Ye | `[0.0, 0.05]` | 24 | ~3.5 h |
| **B** | **Alice** | `[0.10, 0.20]` | 24 | ~3.5 h |
| **C** | Lyric | `[0.30, 0.40]` | 24 | ~3.5 h |

All results save to the **same Drive folder** (`benign_overfitting/N1/`).  
Completed runs are auto-skipped on re-run.

## Step 1 — Environment

In [ ]:
!pip install -q torch torchvision numpy matplotlib seaborn tqdm pandas
import torch
print(f'PyTorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## Step 2 — Mount Drive + Clone repo

In [ ]:
import os, sys
from google.colab import drive

# ── Config ───────────────────────────────────────────────────────────────────
TOKEN     = 'ghp_你的token'    # ← replace with your PAT
REPO_DIR  = '/content/project-6699'
RESULT_DIR = '/content/drive/MyDrive/benign_overfitting/N1'

# ── Mount Drive ───────────────────────────────────────────────────────────────
drive.mount('/content/drive')
os.makedirs(RESULT_DIR, exist_ok=True)
print(f'Results → {RESULT_DIR}')

# ── Clone / update repo ───────────────────────────────────────────────────────
REPO_URL = f'https://{TOKEN}@github.com/alice20030504/EECS-6699.git'
if not os.path.exists(REPO_DIR):
    os.system(f'git clone --branch yixuan {REPO_URL} {REPO_DIR}')
else:
    os.system(f'git -C {REPO_DIR} checkout yixuan')
    os.system(f'git -C {REPO_DIR} pull')

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print('Files:', [f for f in os.listdir('.') if f.endswith('.py')])

## Step 3 — Set your noise rates

**Each account only changes this one line.**

In [ ]:
from run_n1 import N1_CONFIG, run_n1, plot_n1
import copy

cfg = copy.deepcopy(N1_CONFIG)

# ── SET YOUR NOISE RATES HERE ─────────────────────────────────────────────────
# MY_NOISE_RATES = [0.0, 0.05]       # Account A — Ye
MY_NOISE_RATES = [0.10, 0.20]        # Account B — Alice
# MY_NOISE_RATES = [0.30, 0.40]      # Account C — Lyric
# MY_NOISE_RATES = None              # All (single account, ~10 h)

active = MY_NOISE_RATES if MY_NOISE_RATES else cfg['noise_rates']
n_runs = len(cfg['widths']) * len(active) * len(cfg['seeds'])
print(f"This account will train {n_runs} runs")
print(f"Noise rates: {[f'{η:.0%}' for η in active]}")
print(f"Widths:      {cfg['widths']}")

## Step 4 — Run N1

In [ ]:
# Keep-alive thread
import threading, time
def _keep_alive():
    while True:
        time.sleep(60)
        try:
            from google.colab.output import eval_js
            eval_js('0')
        except Exception:
            pass
threading.Thread(target=_keep_alive, daemon=True).start()

results = run_n1(cfg, RESULT_DIR, noise_rates=MY_NOISE_RATES, resume=True)
print(f'\nDone. {len(results)} runs saved to {RESULT_DIR}')

## Step 5 — Plot (run AFTER all 4 accounts finish)

Run this cell only when **all 72 runs** are in the Drive folder.  
It generates Fig 3, Fig 4, Fig 5, and the benign boundary fit.

In [ ]:
plot_n1(RESULT_DIR, cfg)

# Display all figures inline
from IPython.display import Image, display
from pathlib import Path
for fig_name in ['fig3_n1_heatmap.png', 'fig4_n1_dd_overlay.png',
                 'fig5_n1_phase_diagram.png', 'fig5b_benign_boundary.png']:
    p = Path(RESULT_DIR) / fig_name
    if p.exists():
        print(f'\n--- {fig_name} ---')
        display(Image(str(p)))

## Step 6 — Summary table

In [ ]:
import pandas as pd, json
from src.io_utils import load_results
from pathlib import Path

results = load_results(RESULT_DIR, pattern='n1_*.json')
df = pd.DataFrame([{
    'k':       r['width_multiplier'],
    'eta':     f"{r['noise_rate']:.0%}",
    'seed':    r['seed'],
    'n_params': r['n_params'],
    'train_err': f"{r['train_error']:.3f}",
    'test_err':  f"{r['test_error']:.3f}",
} for r in results]).sort_values(['eta', 'k', 'seed'])
print(df.to_string(index=False))

# Show boundary fit if available
fit_path = Path(RESULT_DIR) / 'benign_boundary_fit.json'
if fit_path.exists():
    with open(fit_path) as f:
        fit = json.load(f)
    print(f"\nBenign boundary: {fit['formula']}")